In [18]:
import glob
import os
import json
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from tqdm import tqdm
from pathlib import Path
from red_gym_env_v2 import RedGymEnv
import glob
from stable_baselines3 import PPO
from pathlib import Path
states = glob.glob("battle_states/Test/*.state")
print(states)

['battle_states/Test/battle_Gym5.state', 'battle_states/Test/battle_Rival7.state', 'battle_states/Test/battle_Rival5.state', 'battle_states/Test/battle_Elite41.state', 'battle_states/Test/battle_Rival2.state', 'battle_states/Test/battle_Rival4.state', 'battle_states/Test/battle_Misty.state', 'battle_states/Test/battle_Brock.state']


In [19]:
from pyboy import PyBoy

for s in states:
    pb = PyBoy("/home/di-lab1/PokemonRedExperiments/PokemonRed.gb", window="null")
    with open(s, "rb") as f:
        pb.load_state(f)

    battle_type = pb.memory[0xD057]
    print(s, "battle_type:", battle_type)
    
    pb.stop()

battle_states/Test/battle_Gym5.state battle_type: 2
battle_states/Test/battle_Rival7.state battle_type: 2
battle_states/Test/battle_Rival5.state battle_type: 2
battle_states/Test/battle_Elite41.state battle_type: 2
battle_states/Test/battle_Rival2.state battle_type: 2
battle_states/Test/battle_Rival4.state battle_type: 2
battle_states/Test/battle_Misty.state battle_type: 2
battle_states/Test/battle_Brock.state battle_type: 2


In [20]:
import os
print(os.getcwd())

/home/di-lab1/PokemonRedExperiments/v2


In [21]:
def test_state_health(state_path, ckpt):
    """Run a single quick episode to check if a test state is valid."""

    model = PPO.load(ckpt)

    env_config = {
        "headless": True,
        "save_final_state": False,
        "early_stop": True,
        "action_freq": 60,
        "max_steps": 400,   # keep fast
        "print_rewards": False,
        "save_video": False,
        "fast_video": False,
        "session_path": Path("battle_runs"),
        "gb_path": "/home/di-lab1/PokemonRedExperiments/PokemonRed.gb",
        "debug": False,
        "init_state": state_path
    }

    env = RedGymEnv(env_config)

    obs, info = env.reset()

    done = False
    steps = 0
    ep_reward = 0.0

    while not done and steps < 400:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        ep_reward += reward
        steps += 1

    env.close()

    return {
        "steps": steps,
        "terminated_by": info.get("terminated_by"),
        "won": info.get("won"),
        "reward": ep_reward
    }

In [22]:
state_health = {}
ckpt = "runs/poke_264765440_steps.zip"

for state in states:
    print("Testing:", state)
    result = test_state_health(state, ckpt)
    print(result, "\n")
    state_health[state] = result

Testing: battle_states/Test/battle_Gym5.state
{'steps': 400, 'terminated_by': None, 'won': None, 'reward': -150.0} 

Testing: battle_states/Test/battle_Rival7.state
{'steps': 400, 'terminated_by': None, 'won': None, 'reward': -150.0} 

Testing: battle_states/Test/battle_Rival5.state
{'steps': 400, 'terminated_by': None, 'won': None, 'reward': -150.0} 

Testing: battle_states/Test/battle_Elite41.state
{'steps': 400, 'terminated_by': None, 'won': None, 'reward': -150.0} 

Testing: battle_states/Test/battle_Rival2.state
{'steps': 400, 'terminated_by': None, 'won': None, 'reward': -150.0} 

Testing: battle_states/Test/battle_Rival4.state
{'steps': 400, 'terminated_by': None, 'won': None, 'reward': -149.999922993596} 

Testing: battle_states/Test/battle_Misty.state
{'steps': 400, 'terminated_by': None, 'won': None, 'reward': -150.0} 

Testing: battle_states/Test/battle_Brock.state
{'steps': 126, 'terminated_by': None, 'won': None, 'reward': -100.0} 

